# Multi-Layer Perceptron from Scratch

Implement and train a two-layer classifier in NumPy:

`784 inputs → 128 ReLU units → 10 output logits`

This notebook covers He initialization, stable softmax and cross-entropy, backpropagation, a numerical gradient check, mini-batch SGD, and final test evaluation.

TensorFlow is used only to fetch MNIST. The model's forward pass, loss, gradients, and updates are implemented in NumPy.

In [1]:
import time
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from tensorflow.keras.datasets import mnist

SEED = 42
rng = np.random.default_rng(SEED)

# Find the repository root so this notebook can be launched from the project
# directory or from notebooks/.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find pyproject.toml. Start Jupyter inside the project directory."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"NumPy version: {np.__version__}")

Project root: c:\Users\hp\Documents\GitHub\multilayer-perceptron-from-scratch
NumPy version: 2.2.6


## Load data and create a validation split

Use only the official training split to create train and validation subsets.
The official test split is reserved for the final evaluation.

In [2]:
(X_pool, y_pool), (X_test, y_test) = mnist.load_data()

assert X_pool.shape == (60_000, 28, 28)
assert X_test.shape == (10_000, 28, 28)
assert set(np.unique(y_pool)) == set(range(10))
assert set(np.unique(y_test)) == set(range(10))

permutation = rng.permutation(len(y_pool))
validation_size = 5_000

val_indices = permutation[:validation_size]
train_indices = permutation[validation_size:]

# Track training metrics on a fixed subset to reduce per-epoch evaluation time.
train_metric_indices = train_indices[:10_000].copy()

print(f"Training examples:   {len(train_indices):,}")
print(f"Validation examples: {len(val_indices):,}")
print(f"Test examples:       {len(y_test):,}")

Training examples:   55,000
Validation examples: 5,000
Test examples:       10,000


## Stable output probabilities and cross-entropy

For logits `z`, softmax is computed after subtracting the largest logit in each row.
This is mathematically equivalent to ordinary softmax and helps prevent overflow.

For softmax combined with cross-entropy, the derivative with respect to the logits is
`(probabilities - one_hot_labels) / batch_size`.

In [3]:
def softmax(logits: np.ndarray) -> np.ndarray:
    """Convert each row of logits into a numerically stable probability vector."""
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=1, keepdims=True)


def cross_entropy_from_logits(
    logits: np.ndarray,
    labels: np.ndarray,
) -> float:
    """Mean multiclass cross-entropy, computed directly from logits."""
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    log_normalizer = np.log(np.sum(np.exp(shifted), axis=1))
    row_indices = np.arange(labels.shape[0])
    losses = log_normalizer - shifted[row_indices, labels]
    return float(np.mean(losses))

## MLP implementation

For a batch `X`, the forward pass is:

1. `Z1 = X @ W1 + b1`
2. `A1 = ReLU(Z1)`
3. `logits = A1 @ W2 + b2`

Backpropagation applies the chain rule from the cross-entropy gradient at the output
back through the output layer and ReLU hidden layer.

In [4]:
class NumpyMLP:
    """One-hidden-layer classifier implemented with NumPy."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        seed: int = 42,
        dtype: np.dtype = np.float32,
    ) -> None:
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.dtype = np.dtype(dtype)

        init_rng = np.random.default_rng(seed)

        # He initialization for ReLU hidden units; Xavier-style scaling for logits.
        self.params = {
            "W1": (
                init_rng.standard_normal((input_dim, hidden_dim)).astype(self.dtype)
                * np.sqrt(2.0 / input_dim)
            ),
            "b1": np.zeros(hidden_dim, dtype=self.dtype),
            "W2": (
                init_rng.standard_normal((hidden_dim, output_dim)).astype(self.dtype)
                * np.sqrt(1.0 / hidden_dim)
            ),
            "b2": np.zeros(output_dim, dtype=self.dtype),
        }

    def forward(
        self,
        X: np.ndarray,
    ) -> tuple[np.ndarray, tuple[np.ndarray, np.ndarray, np.ndarray]]:
        """Return logits and the intermediate values needed for backpropagation."""
        W1, b1 = self.params["W1"], self.params["b1"]
        W2, b2 = self.params["W2"], self.params["b2"]

        Z1 = X @ W1 + b1
        A1 = np.maximum(Z1, 0)
        logits = A1 @ W2 + b2

        return logits, (X, Z1, A1)

    def loss(self, X: np.ndarray, labels: np.ndarray) -> float:
        logits, _ = self.forward(X)
        return cross_entropy_from_logits(logits, labels)

    def backward(
        self,
        labels: np.ndarray,
        cache: tuple[np.ndarray, np.ndarray, np.ndarray],
    ) -> dict[str, np.ndarray]:
        """Compute mean-loss gradients for all weights and biases."""
        X, Z1, A1 = cache
        batch_size = labels.shape[0]

        logits = A1 @ self.params["W2"] + self.params["b2"]
        d_logits = softmax(logits)
        d_logits[np.arange(batch_size), labels] -= 1.0
        d_logits /= batch_size

        dW2 = A1.T @ d_logits
        db2 = np.sum(d_logits, axis=0)

        d_hidden = d_logits @ self.params["W2"].T
        d_Z1 = d_hidden * (Z1 > 0)

        dW1 = X.T @ d_Z1
        db1 = np.sum(d_Z1, axis=0)

        return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

## Verify backpropagation with finite differences

Compare analytical gradients with centered finite-difference estimates on a tiny network.
This is intentionally small so every parameter can be checked quickly.

In [5]:
small_model = NumpyMLP(
    input_dim=4,
    hidden_dim=5,
    output_dim=3,
    seed=7,
    dtype=np.float64,
)

X_small = np.array(
    [
        [0.2, 0.1, 0.7, 0.4],
        [0.9, 0.3, 0.2, 0.5],
    ],
    dtype=np.float64,
)
y_small = np.array([2, 0], dtype=np.int64)

_, cache = small_model.forward(X_small)
analytic = small_model.backward(y_small, cache)

epsilon = 1e-5
max_relative_error = 0.0

for name, parameter in small_model.params.items():
    numerical = np.zeros_like(parameter)

    for index in np.ndindex(parameter.shape):
        original_value = parameter[index]

        parameter[index] = original_value + epsilon
        loss_plus = small_model.loss(X_small, y_small)

        parameter[index] = original_value - epsilon
        loss_minus = small_model.loss(X_small, y_small)

        parameter[index] = original_value
        numerical[index] = (loss_plus - loss_minus) / (2.0 * epsilon)

    denominator = np.maximum(
        1e-12,
        np.abs(analytic[name]) + np.abs(numerical),
    )
    relative_error = np.abs(analytic[name] - numerical) / denominator
    max_relative_error = max(max_relative_error, float(relative_error.max()))

print(f"Maximum relative gradient error: {max_relative_error:.2e}")
assert max_relative_error < 1e-5, "Gradient check failed."
print("Gradient check passed.")

Maximum relative gradient error: 1.08e-10
Gradient check passed.


## Train with mini-batch SGD

Images are flattened and normalized batch-by-batch to avoid keeping large float32
copies of the entire dataset in memory. Parameters are updated after each mini-batch.

In [6]:
def make_batch(
    images: np.ndarray,
    labels: np.ndarray,
    indices: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Select, flatten, and normalize an image batch."""
    X_batch = images[indices].reshape(len(indices), -1).astype(np.float32)
    X_batch /= np.float32(255.0)
    y_batch = labels[indices].astype(np.int64)
    return X_batch, y_batch


def evaluate(
    model: NumpyMLP,
    images: np.ndarray,
    labels: np.ndarray,
    indices: np.ndarray,
    batch_size: int = 512,
) -> tuple[float, float]:
    """Return mean cross-entropy and accuracy over the selected indices."""
    total_loss = 0.0
    total_correct = 0

    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start : start + batch_size]
        X_batch, y_batch = make_batch(images, labels, batch_indices)

        logits, _ = model.forward(X_batch)
        batch_count = len(batch_indices)

        total_loss += cross_entropy_from_logits(logits, y_batch) * batch_count
        total_correct += int(np.sum(np.argmax(logits, axis=1) == y_batch))

    return total_loss / len(indices), total_correct / len(indices)


INPUT_DIM = 28 * 28
HIDDEN_DIM = 128
OUTPUT_DIM = 10

EPOCHS = 10
BATCH_SIZE = 128
LEARNING_RATE = 0.1

model = NumpyMLP(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    seed=SEED,
    dtype=np.float32,
)

history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    rng.shuffle(train_indices)

    for start in range(0, len(train_indices), BATCH_SIZE):
        batch_indices = train_indices[start : start + BATCH_SIZE]
        X_batch, y_batch = make_batch(X_pool, y_pool, batch_indices)

        _, cache = model.forward(X_batch)
        gradients = model.backward(y_batch, cache)

        for name, parameter in model.params.items():
            parameter -= LEARNING_RATE * gradients[name]

    train_loss, train_accuracy = evaluate(
        model, X_pool, y_pool, train_metric_indices
    )
    val_loss, val_accuracy = evaluate(model, X_pool, y_pool, val_indices)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    elapsed = time.perf_counter() - epoch_start
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f}, acc {train_accuracy:.4f} | "
        f"val loss {val_loss:.4f}, acc {val_accuracy:.4f} | "
        f"{elapsed:.1f}s"
    )

Epoch 01/10 | train loss 0.3084, acc 0.9166 | val loss 0.3326, acc 0.9058 | 1.2s
Epoch 02/10 | train loss 0.2474, acc 0.9329 | val loss 0.2683, acc 0.9208 | 1.2s
Epoch 03/10 | train loss 0.2071, acc 0.9440 | val loss 0.2305, acc 0.9308 | 1.2s
Epoch 04/10 | train loss 0.1820, acc 0.9497 | val loss 0.2052, acc 0.9400 | 1.2s
Epoch 05/10 | train loss 0.1556, acc 0.9578 | val loss 0.1800, acc 0.9472 | 1.4s
Epoch 06/10 | train loss 0.1399, acc 0.9614 | val loss 0.1640, acc 0.9518 | 1.3s
Epoch 07/10 | train loss 0.1262, acc 0.9666 | val loss 0.1501, acc 0.9566 | 1.4s
Epoch 08/10 | train loss 0.1140, acc 0.9689 | val loss 0.1399, acc 0.9590 | 1.2s
Epoch 09/10 | train loss 0.1039, acc 0.9712 | val loss 0.1295, acc 0.9638 | 1.3s
Epoch 10/10 | train loss 0.0975, acc 0.9736 | val loss 0.1244, acc 0.9654 | 1.3s


## Training curves

Training metrics are measured on a fixed 10,000-example subset; validation metrics use
the full validation split. A widening gap between training and validation performance
can indicate overfitting.

In [7]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=history["epoch"],
        y=history["train_loss"],
        mode="lines+markers",
        name="Training loss (10k sample)",
    )
)
fig.add_trace(
    go.Scatter(
        x=history["epoch"],
        y=history["val_loss"],
        mode="lines+markers",
        name="Validation loss",
    )
)
fig.update_layout(
    title="Training and validation loss",
    xaxis_title="Epoch",
    yaxis_title="Cross-entropy loss",
)
fig.show()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=history["epoch"],
        y=history["train_accuracy"],
        mode="lines+markers",
        name="Training accuracy (10k sample)",
    )
)
fig.add_trace(
    go.Scatter(
        x=history["epoch"],
        y=history["val_accuracy"],
        mode="lines+markers",
        name="Validation accuracy",
    )
)
fig.update_layout(
    title="Training and validation accuracy",
    xaxis_title="Epoch",
    yaxis_title="Accuracy",
    yaxis=dict(tickformat=".0%"),
)
fig.show()

## Final test evaluation and model artifact

Evaluate the official test split once, after training decisions are complete.
Save the small NumPy parameter arrays for use by the Streamlit app.

In [8]:
test_indices = np.arange(len(y_test))
test_loss, test_accuracy = evaluate(model, X_test, y_test, test_indices)

print(f"Final test loss:     {test_loss:.4f}")
print(f"Final test accuracy: {test_accuracy:.2%}")

model_dir = PROJECT_ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)
artifact_path = model_dir / "mnist_mlp.npz"

np.savez_compressed(
    artifact_path,
    W1=model.params["W1"],
    b1=model.params["b1"],
    W2=model.params["W2"],
    b2=model.params["b2"],
    input_dim=np.array(model.input_dim),
    hidden_dim=np.array(model.hidden_dim),
    output_dim=np.array(model.output_dim),
    pixel_scale=np.array(255.0, dtype=np.float32),
    test_accuracy=np.array(test_accuracy, dtype=np.float32),
)

print(f"Saved model parameters to: {artifact_path}")

Final test loss:     0.1160
Final test accuracy: 96.67%
Saved model parameters to: c:\Users\hp\Documents\GitHub\multilayer-perceptron-from-scratch\models\mnist_mlp.npz


## Takeaways

- The MLP's forward pass, cross-entropy, and backpropagation are implemented in NumPy.
- The finite-difference check validates the analytical gradients on a small network.
- Training and validation curves help assess convergence and overfitting.
- The test score is a final estimate of generalization, not a tuning signal.
- `models/mnist_mlp.npz` contains the trained weights and architecture dimensions for the Streamlit demo.

Next: compare architectures and training settings in `02_Training_Experiments.ipynb`.